In [ ]:
# Example 1: Inspect the typer app exposed by fastapi_new.cli
import typer

from fastapi_new.cli import app, main

assert isinstance(app, typer.Typer), f"Expected typer.Typer, got {type(app)}"
assert callable(main), "fastapi_new.cli.main must be callable"

# The CLI registers a single command (`new`) on the Typer app.
registered = [cmd.name or cmd.callback.__name__ for cmd in app.registered_commands]
print(f"Registered commands: {registered}")
assert len(app.registered_commands) >= 1


In [ ]:
# Example 2: ProjectConfig dataclass round-trip
import pathlib
from dataclasses import fields, is_dataclass

from fastapi_new.new import ProjectConfig

assert is_dataclass(ProjectConfig)
field_names = {f.name for f in fields(ProjectConfig)}
assert {"name", "path", "python"}.issubset(field_names), field_names

cfg = ProjectConfig(name="demo", path=pathlib.Path("/tmp/demo"), python="3.12")
assert cfg.name == "demo"
assert cfg.path == pathlib.Path("/tmp/demo")
assert cfg.python == "3.12"
print(cfg)


In [ ]:
# Example 3: Python-version validator (>= 3.10 required)
from fastapi_new.new import _validate_python_version

# None / empty -> None (no error)
assert _validate_python_version(None) is None
assert _validate_python_version("") is None

# Too-old versions return an error string
err = _validate_python_version("3.9")
assert err is not None and "3.10 or higher" in err, err
err = _validate_python_version("2.7")
assert err is not None

# Supported versions return None
assert _validate_python_version("3.10") is None
assert _validate_python_version("3.12") is None
assert _validate_python_version("3.14") is None

# Malformed -> None (let uv handle it)
assert _validate_python_version("weird") is None
print("version validator OK")


In [ ]:
# Example 4: README generator + template constant inspection
from fastapi_new.new import TEMPLATE_CONTENT, _generate_readme

readme = _generate_readme("my_project")
assert isinstance(readme, str)
assert readme.startswith("# my_project")
assert "FastAPI" in readme
assert "uv run fastapi dev" in readme

assert "from fastapi import FastAPI" in TEMPLATE_CONTENT
assert "app = FastAPI()" in TEMPLATE_CONTENT
print(f"README length: {len(readme)} chars; TEMPLATE length: {len(TEMPLATE_CONTENT)} chars")


In [ ]:
# Example 5: Drive the installed `fastapi-new` console script end-to-end (--help only; no
# project is created). Two notes:
# - fastapi_new's Typer app is a SINGLE command — the app IS `new` (Usage: fastapi-new
#   [OPTIONS] [PROJECT]); there is no `new` subcommand to invoke.
# - Subprocess, not typer.testing.CliRunner: inside a Jupyter kernel typer/rich render help
#   through Jupyter's display hooks, so CliRunner's stdout capture comes back empty.
# NO_COLOR removes color but rich still emits bold/style ANSI codes mid-line, so strip
# escapes before matching.
import os
import re
import subprocess

proc = subprocess.run(
    ["fastapi-new", "--help"],
    capture_output=True,
    text=True,
    timeout=120,
    env={**os.environ, "NO_COLOR": "1", "TERM": "dumb"},
)

assert proc.returncode == 0, f"--help exited non-zero: {proc.returncode}\n{proc.stderr}"
help_text = re.sub(r"\x1b\[[0-9;]*m", "", proc.stdout).lower()
print(help_text[:400])
assert "usage: fastapi-new" in help_text, help_text[:200]
assert "--python" in help_text
assert "project" in help_text
print("CLI help OK")
